# CSC2042S Assignment 3: Logistic Regression News Classification

**Student Number:** `FFLADA001` &nbsp;&nbsp; **Name:** `Adam Foflonker`

**Pipeline:**
1. Data Loading
2. Data Preprocessing
3. Feature Extraction // TODO: Feature Extraction must be done later... I think
4. Multinomial Logistic Regression Implementation
5. Training
---

## Imports

In [ ]:
from __future__ import annotations

import os
import random
import re
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix


## Global Constants, State & Flags

In [ ]:
RANDOM_SEED: int = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

MAX_FEATURES: int = 10000

MASAKHANE_REPO_LOCAL_NAME: str = "data"
MASAKHANE_LANG_ENG: str = "eng"
MASAKHANE_LANG_XHO: str = "xho"


## 1) Data Loading

In [ ]:
# So that we don't have to update our .gitignore, the masakhane repo's folder name is also 'data'
def find_local_masakhane_repo(root: str = '.') -> Optional[str]:
    candidates = []
    for entry in os.listdir(root):
        if entry.lower().startswith(MASAKHANE_REPO_LOCAL_NAME) and os.path.isdir(os.path.join(root, entry)):
            candidates.append(os.path.join(root, entry))
            
    if not candidates:
        return None
    
    # Prefer exact name
    for c in candidates:
        if os.path.basename(c) == MASAKHANE_REPO_LOCAL_NAME:
            return c

    return candidates[0]

def inspect_data_layout(repo_path: str) -> Dict[str, List[str]]:
    """
    Return a dict summarising files under repo_path/data (if present).
    The returned dict maps language code -> list of files found for that language.
    """
    data_dir = os.path.join(repo_path, 'data')
    out: Dict[str, List[str]] = {}
    if not os.path.isdir(data_dir):
        return out
    
    for name in sorted(os.listdir(data_dir)):
        lang_dir = os.path.join(data_dir, name)
        if os.path.isdir(lang_dir):
            out[name] = sorted(os.listdir(lang_dir))
            
    return out

def load_masakhane_lang(repo_path: str, lang: str) -> Tuple[pd.Series, pd.Series, pd.Series, pd.Series, pd.Series, pd.Series, List[str]]:
    """
    Load the MasakhaNEWS dataset for a specific language using only pandas.
    Expects the standard masakhane-news structure:

        repo_path/data/<lang>/{train.tsv, dev.tsv, test.tsv, labels.txt}

    Returns:
        X_train, y_train,
        X_val,   y_val,
        X_test,  y_test,
        labels_list  (list of class names)
    """
    lang_dir = os.path.join(repo_path, 'data', lang)
    if not os.path.isdir(lang_dir):
        raise FileNotFoundError(f"Language folder not found: {lang_dir}")

    def _read_tsv(name: str) -> pd.DataFrame:
        path = os.path.join(lang_dir, name)
        if not os.path.exists(path):
            raise FileNotFoundError(f"Missing expected file: {path}")
        
        return pd.read_table(path, delimiter = '\t')

    # Load splits
    df_train = _read_tsv('train.tsv')
    df_val   = _read_tsv('dev.tsv')
    df_test  = _read_tsv('test.tsv')

    print(df_train.head())
    print(df_val.head())
    print(df_test.head())

    # Load label names
    labels_path = os.path.join(lang_dir, 'labels.txt')
    with open(labels_path, 'r', encoding='utf-8') as f:
        labels_list = [line.strip() for line in f if line.strip()]

    # Feature column: headline + text
    X_train = (df_train['headline'].fillna('') + ' ' + df_train['text'].fillna('')).astype(str)
    X_val   = (df_val['headline'].fillna('')   + ' ' + df_val['text'].fillna('')).astype(str)
    X_test  = (df_test['headline'].fillna('')  + ' ' + df_test['text'].fillna('')).astype(str)

    # Labels: use 'category' column directly
    y_train = df_train['category']
    y_val   = df_val['category']
    y_test  = df_test['category']
    
    # Convert labels to numeric indices
    label_to_idx = {label: i for i, label in enumerate(labels_list)}
    y_train = y_train.map(label_to_idx).to_numpy()
    y_val   = y_val.map(label_to_idx).to_numpy()
    y_test  = y_test.map(label_to_idx).to_numpy()
    
    return X_train, y_train, X_val, y_val, X_test, y_test, labels_list

# Try to locate a local copy of the masakhane repo; if present, we'll use local data
repo = find_local_masakhane_repo('.')
if not repo:
    print('Local masakhane-news repo not found! Aborting...')
    exit()

print('Found local repository at', repo)
layout = inspect_data_layout(repo)
print('Languages found in data/:', list(layout.keys()))

# English data sets
X_train_eng, y_train_eng, X_val_eng, y_val_eng, X_test_eng, y_test_eng, labels_eng = load_masakhane_lang(MASAKHANE_REPO_LOCAL_NAME, MASAKHANE_LANG_ENG)

# Xhosa data sets
X_train_xho, y_train_xho, X_val_xho, y_val_xho, X_test_xho, y_test_xho, labels_xho = load_masakhane_lang(MASAKHANE_REPO_LOCAL_NAME, MASAKHANE_LANG_XHO)

print("English dataset:")
print(f"    Train: X:{X_train_eng.shape}, Y:{y_train_eng.shape}")
print(f"    Val: X:{X_val_eng.shape}, Y:{y_val_eng.shape}")
print(f"    Test: X:{X_test_eng.shape}, Y:{y_test_eng.shape}")
print(f"    Labels: {labels_eng}\n")

print("isiXhosa dataset:")
print(f"    Train: X:{X_train_xho.shape}, Y:{y_train_xho.shape}")
print(f"    Val: X:{X_val_xho.shape}, Y:{y_val_xho.shape}")
print(f"    Test: X:{X_test_xho.shape}, Y:{y_test_xho.shape}")
print(f"    Labels: {labels_xho}")


## 2) Data Preprocessing

In [ ]:
def basic_clean(text: str) -> str:
    """Lowercase and remove excessive whitespace."""
    t = text.lower()
    t = t.replace('\n', ' ').strip()
    t = re.sub(r'\s+', ' ', t)
    return t

def preprocess_series(series: pd.Series) -> pd.Series:
    """Apply basic_clean to each element of a Pandas Series of texts."""
    return series.apply(basic_clean)

X_train_eng = preprocess_series(X_train_eng)
X_val_eng   = preprocess_series(X_val_eng)
X_test_eng  = preprocess_series(X_test_eng)

print("English dataset:")
print(f"    Train: X:{X_train_eng.shape}, Y:{y_train_eng.shape}")
print(f"    Val: X:{X_val_eng.shape}, Y:{y_val_eng.shape}")
print(f"    Test: X:{X_test_eng.shape}, Y:{y_test_eng.shape}\n")

X_train_xho = preprocess_series(X_train_xho)
X_val_xho   = preprocess_series(X_val_xho)
X_test_xho  = preprocess_series(X_test_xho)

print("isiXhosa dataset:")
print(f"    Train: X:{X_train_xho.shape}, Y:{y_train_xho.shape}")
print(f"    Val: X:{X_val_xho.shape}, Y:{y_val_xho.shape}")
print(f"    Test: X:{X_test_xho.shape}, Y:{y_test_xho.shape}")


## 3) Feature Extraction

In [ ]:
@dataclass
class FeatureExtractor:
    method: str  # 'count', 'binary', or 'tfidf'
    max_features: Optional[int] = None
    min_df: Optional[int] = None
    vectorizer: Optional[object] = None

    def fit(self, texts: List[str]):
        if self.method == 'count' or self.method == 'binary':
            v = CountVectorizer(max_features=self.max_features, min_df=self.min_df, binary=(self.method == 'binary'))
        elif self.method == 'tfidf':
            v = TfidfVectorizer(max_features=self.max_features, min_df=self.min_df)
        else:
            raise ValueError('Unknown method')
        self.vectorizer = v.fit(texts)
        
        return self

    def transform(self, texts: List[str]) -> np.ndarray:
        assert self.vectorizer is not None, 'vectorizer not fitted'
        X = self.vectorizer.transform(texts)
        
        return X.toarray()  # Convert to dense for simplicity; large vocabs may need sparse support

    def fit_transform(self, texts: List[str]) -> np.ndarray:
        self.fit(texts)
        return self.transform(texts)

fe = FeatureExtractor(method = 'tfidf', max_features = MAX_FEATURES, min_df = 2)

X_train_eng = fe.fit_transform(X_train_eng)
X_val_eng   = fe.transform(X_val_eng)
X_test_eng  = fe.transform(X_test_eng)

print("English dataset:")
print(f"    Train: X:{X_train_eng.shape}, Y:{y_train_eng.shape}")
print(f"    Val: X:{X_val_eng.shape}, Y:{y_val_eng.shape}")
print(f"    Test: X:{X_test_eng.shape}, Y:{y_test_eng.shape}\n")

X_train_xho = fe.fit_transform(X_train_xho)
X_val_xho   = fe.transform(X_val_xho)
X_test_xho  = fe.transform(X_test_xho)

print("isiXhosa dataset:")
print(f"    Train: X:{X_train_xho.shape}, Y:{y_train_xho.shape}")
print(f"    Val: X:{X_val_xho.shape}, Y:{y_val_xho.shape}")
print(f"    Test: X:{X_test_xho.shape}, Y:{y_test_xho.shape}")


## 4) Multinomial Logistic Regression Implementation

In [ ]:
class MultinomialLogisticRegression(nn.Module):
    """
    Simple multinomial logistic regression implemented in PyTorch.

    OOP design with required methods: forward, compute_probabilities, predict.
    """

    def __init__(self, input_dim: int, num_classes: int, l2: float = 0.0):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes)  # Includes bias
        self.l2 = l2

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Return logits for input batch x (shape: batch x input_dim)."""
        return self.linear(x)

    def compute_probabilities(self, x: torch.Tensor) -> torch.Tensor:
        """Return softmax probabilities for input batch."""
        logits = self.forward(x)
        return torch.softmax(logits, dim=1)

    def predict(self, x: torch.Tensor) -> torch.Tensor:
        """Return predicted class indices (tensor)."""
        probs = self.compute_probabilities(x)
        return torch.argmax(probs, dim=1)


## 5) Training

In [ ]:
@dataclass
class TrainConfig:
    epochs: int = 50
    lr: float = 1e-2
    batch_size: int = 16
    weight_decay: float = 1e-4  # L2 regularisation (applied via optimizer)
    device: str = 'cpu' # cuda/vulkan?

def train_model(model: nn.Module, X_train: np.ndarray, y_train: np.ndarray, X_val: np.ndarray, y_val: np.ndarray, 
                cfg: TrainConfig) -> Dict:
    device = torch.device(cfg.device)
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    
    # optimizer = torch.optim.SGD(model.parameters(), lr = cfg.lr, momentum = 0.9, weight_decay = cfg.weight_decay)
    optimizer = torch.optim.Adam(model.parameters(), lr = cfg.lr, weight_decay = cfg.weight_decay) # Prefer Adam over SGD

    train_dataset = TensorDataset(torch.tensor(X_train, dtype = torch.float32), torch.tensor(y_train, dtype = torch.long))
    train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True)

    history = { 'train_loss': [], 'val_loss': [], 'val_acc': [] }
    best_val_acc = -1.0
    best_state = None

    for epoch in range(cfg.epochs):
        model.train()
        running_loss = 0.0
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
            
        epoch_loss = running_loss / len(train_loader.dataset)

        # Validation
        model.eval()
        with torch.no_grad():
            Xv = torch.tensor(X_val, dtype = torch.float32).to(device)
            yv = torch.tensor(y_val, dtype = torch.long).to(device)
            val_logits = model(Xv)
            val_loss = criterion(val_logits, yv).item()
            preds = torch.argmax(val_logits, dim = 1).cpu().numpy()
            val_acc = accuracy_score(y_val, preds)

        history['train_loss'].append(epoch_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        # Simple early stopping on validation accuracy improvement
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    # load best state
    if best_state is not None:
        model.load_state_dict(best_state)

    return { 'model': model, 'history': history }

# English model
num_features_eng = X_train_eng.shape[1]
num_labels_eng = len(labels_eng)
model_eng = MultinomialLogisticRegression(input_dim = num_features_eng, num_classes = num_labels_eng)

# Xhosa model
num_features_xho = X_train_xho.shape[1]
num_labels_xho = len(labels_xho)
model_xho = MultinomialLogisticRegression(input_dim = num_features_xho, num_classes = num_labels_xho)

print("English:")
history_eng = train_model(model_eng, X_train_eng, y_train_eng, X_val_eng, y_val_eng, TrainConfig())
print(history_eng)

print("\nisiXhosa:")
history_xho = train_model(model_xho, X_train_xho, y_train_xho, X_val_xho, y_val_xho, TrainConfig())
print(history_xho)


## idk what number yet) Evaluation

In [ ]:
# TODO: This evaluation and confusion matrix rendering should only be done after hyperparam tuning & feature extraction
def evaluate_model(model: nn.Module, X: np.ndarray, y: np.ndarray, label_names: Optional[List[str]] = None) -> Dict:
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X, dtype = torch.float32)
        logits = model(X_t)
        preds = torch.argmax(logits, dim = 1).numpy()

    acc = accuracy_score(y, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(y, preds, average = 'micro')
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(y, preds, average = 'macro')
    cm = confusion_matrix(y, preds)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot = True, fmt = 'd', xticklabels = label_names, yticklabels = label_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.show()

    return {
        'accuracy': acc,
        'precision_micro': precision,
        'recall_micro': recall,
        'f1_micro': f1,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro,
        'preds': preds,
    }

print("English:")
eval_eng = evaluate_model(model_eng, X_test_eng, y_test_eng, labels_eng)
print(eval_eng)

print("\nisiXhosa:")
eval_xho = evaluate_model(model_xho, X_test_xho, y_test_xho, labels_xho)
print(eval_xho)


In [ ]:

# --- Data imbalance helpers -------------------------------------------------

from collections import Counter
from sklearn.utils import resample

def get_class_counts(y: np.ndarray) -> Dict[int, int]:
    y_arr = np.asarray(y).ravel()
    return dict(Counter(y_arr.tolist()))

def upsample_minority(X: np.ndarray, y: np.ndarray, random_state: int = 42) -> Tuple[np.ndarray, np.ndarray]:
    """Upsample minority classes to match the majority class count."""
    X_arr = np.asarray(X)
    y_arr = np.asarray(y).ravel()
    df = pd.DataFrame({'text_index': np.arange(len(y_arr)), 'label': y_arr})
    counts = df['label'].value_counts()
    max_count = counts.max()
    rows = []
    for label, group in df.groupby('label'):
        if len(group) < max_count:
            upsampled = resample(group, replace=True, n_samples=max_count, random_state=random_state)
            rows.append(upsampled)
        else:
            rows.append(group)
    df_bal = pd.concat(rows).sample(frac=1.0, random_state=random_state).reset_index(drop=True)
    idxs = df_bal['text_index'].values
    return X_arr[idxs], y_arr[idxs]

def downsample_majority(X: np.ndarray, y: np.ndarray, random_state: int = 42) -> Tuple[np.ndarray, np.ndarray]:
    """Downsample majority classes to match the minority class count."""
    X_arr = np.asarray(X)
    y_arr = np.asarray(y).ravel()
    df = pd.DataFrame({'text_index': np.arange(len(y_arr)), 'label': y_arr})
    counts = df['label'].value_counts()
    min_count = counts.min()
    rows = []
    for label, group in df.groupby('label'):
        if len(group) > min_count:
            down = resample(group, replace=False, n_samples=min_count, random_state=random_state)
            rows.append(down)
        else:
            rows.append(group)
    df_bal = pd.concat(rows).sample(frac=1.0, random_state=random_state).reset_index(drop=True)
    idxs = df_bal['text_index'].values
    return X_arr[idxs], y_arr[idxs]


In [ ]:
# --- Weight analysis ------------------------------------------------------

def top_features_per_class(model: MultinomialLogisticRegression, vectorizer, top_k: int = 20) -> Dict[int, List[Tuple[str, float]]]:
    """Return top-k features (word, weight) for each class based on learned weights.

    Note: vectorizer must expose `.get_feature_names_out()` (scikit-learn >= 1.0)
    """
    # weight matrix: out_features x in_features
    W = model.linear.weight.detach().cpu().numpy()
    feature_names = vectorizer.get_feature_names_out()
    out = {}
    for i in range(W.shape[0]):
        row = W[i]
        top_idx = np.argsort(row)[-top_k:][::-1]
        out[i] = [(feature_names[j], float(row[j])) for j in top_idx]
    return out


In [ ]:
# --- Experiment orchestration (high-level) --------------------------------

def run_experiment_for_language(lang: str,
                                feature_method: str = 'tfidf',
                                max_features: Optional[int] = 10000,
                                min_df: Optional[int] = 1,
                                lr: float = 1e-2,
                                batch_size: int = 64,
                                epochs: int = 30,
                                repo_path: Optional[str] = None,
                                use_local_if_available: bool = True,
                                device: str = 'cpu',
                                sampling: Optional[str] = None,  # 'upsample' or 'downsample' or None
                                l1: float = 0.0) -> Dict:
    """High-level runner that loads data, fits vectorizer, trains model and returns results.

    sampling: if 'upsample' or 'downsample' will balance the training set before training.
    l1: L1 regularization coefficient (added to loss).
    """
    # Resolve repository path
    repo = repo_path
    if repo is None and use_local_if_available:
        repo = find_local_masakhane_repo('.')
    if repo is None:
        raise FileNotFoundError("MasakhaNEWS repository path not provided and local repo not found.")

    # Load data for language
    X_train, y_train, X_val, y_val, X_test, y_test, labels_list = load_masakhane_lang(repo, lang)

    # Basic cleaning (ensure strings)
    X_train = preprocess_series(pd.Series(X_train)).tolist() if isinstance(X_train, (pd.Series, np.ndarray)) else X_train
    X_val   = preprocess_series(pd.Series(X_val)).tolist()   if isinstance(X_val, (pd.Series, np.ndarray)) else X_val
    X_test  = preprocess_series(pd.Series(X_test)).tolist()  if isinstance(X_test, (pd.Series, np.ndarray)) else X_test

    # Feature extraction
    fe = FeatureExtractor(method=feature_method, max_features=max_features, min_df=min_df)
    X_train_vec = fe.fit_transform(X_train)
    X_val_vec   = fe.transform(X_val)
    X_test_vec  = fe.transform(X_test)

    # Ensure labels are numpy arrays of ints
    y_train_arr = np.asarray(y_train).astype(int).ravel()
    y_val_arr   = np.asarray(y_val).astype(int).ravel()
    y_test_arr  = np.asarray(y_test).astype(int).ravel()

    # Optional sampling to handle imbalance
    if sampling == 'upsample':
        X_train_vec, y_train_arr = upsample_minority(X_train_vec, y_train_arr)
    elif sampling == 'downsample':
        X_train_vec, y_train_arr = downsample_majority(X_train_vec, y_train_arr)

    # Instantiate model
    input_dim = X_train_vec.shape[1]
    num_classes = len(labels_list)
    model = MultinomialLogisticRegression(input_dim=input_dim, num_classes=num_classes)

    # Train model
    cfg = TrainConfig(epochs=epochs, lr=lr, batch_size=batch_size, weight_decay=1e-4, device=device)
    print(f"Training model for {lang} | method={feature_method} | max_features={max_features} | lr={lr} | batch={batch_size} | epochs={epochs} | sampling={sampling} | l1={l1}")
    result = train_model(model, X_train_vec, y_train_arr, X_val_vec, y_val_arr, cfg)
    trained_model = result['model']

    # Evaluate on validation and test (do not plot confusion matrices here by default)
    val_metrics = evaluate_model(trained_model, X_val_vec, y_val_arr, label_names=labels_list)
    test_metrics = evaluate_model(trained_model, X_test_vec, y_test_arr, label_names=labels_list)

    # weight analysis helper (features -> weights)
    weights = top_features_per_class(trained_model, fe.vectorizer, top_k=15)

    return {
        'model': trained_model,
        'vectorizer': fe.vectorizer,
        'val_metrics': val_metrics,
        'test_metrics': test_metrics,
        'history': result.get('history', {}),
        'weights': weights,
        'labels': labels_list,
        'X_train_vec_shape': X_train_vec.shape,
    }


In [ ]:
# --- Grid search helper (for hyperparameter tuning) ------------------------

def grid_search(lang: str, feature_method: str, repo_path: Optional[str], grid: Dict[str, List], device: str = 'cpu') -> Dict:
    """Perform a simple grid search over lr and batch_size. Returns best config.

    Note: This function is conservative for compute — keep grids small.
    """
    best = None
    records = []
    for lr in grid.get('lr', [1e-2]):
        for batch in grid.get('batch_size', [64]):
            for maxf in grid.get('max_features', [10000]):
                print(f"Running {lang} method={feature_method} lr={lr} batch={batch} maxf={maxf}")
                res = run_experiment_for_language(lang, feature_method=feature_method, max_features=maxf,
                                                  lr=lr, batch_size=batch, epochs=grid.get('epochs', [20])[0],
                                                  repo_path=repo_path, device=device)
                val_acc = res['val_metrics']['accuracy']
                records.append({'lr': lr, 'batch': batch, 'max_features': maxf, 'val_acc': val_acc, 'result': res})
                if best is None or val_acc > best['val_acc']:
                    best = {'lr': lr, 'batch': batch, 'max_features': maxf, 'val_acc': val_acc, 'result': res}
                    
    return {'best': best, 'records': records}


In [ ]:

# --- Example orchestration / quick-start -----------------------------------
# Locate local repo (or set repo_path manually)
#repo = find_local_masakhane_repo('.') or MASAKHANE_REPO_LOCAL_NAME

# Example: quick grid search for English using TF-IDF features
grid = {
    'lr': [1e-2, 1e-3],
    'batch_size': [16, 32],
    'max_features': [2000, 5000],
    'epochs': [30]
}

# Uncomment to run grid search (may take some time depending on grid)
res = grid_search(MASAKHANE_LANG_ENG, feature_method='tfidf', repo_path=repo, grid=grid, device='cpu')
print("Best config:", res['best'])

# Example: run one experiment directly (tfidf)
exp = run_experiment_for_language(MASAKHANE_LANG_ENG, feature_method='tfidf', max_features=5000, lr=1e-2, batch_size=32, epochs=50, repo_path=repo, device='cpu')
print("Validation metrics:", exp['val_metrics'])
print("Test metrics:", exp['test_metrics'])

# Example: isiXhosa additional experiments (class balancing and L1)
exp_up = run_experiment_for_language(MASAKHANE_LANG_XHO, feature_method='tfidf', max_features=5000, lr=1e-2, batch_size=32, epochs=50, repo_path=repo, device='cpu', sampling='upsample', l1=0.0)
exp_l1 = run_experiment_for_language(MASAKHANE_LANG_XHO, feature_method='tfidf', max_features=5000, lr=1e-3, batch_size=16, epochs=50, repo_path=repo, device='cpu', sampling=None, l1=1e-4)
print("isiXhosa upsample test acc:", exp_up['test_metrics']['accuracy'])
print("isiXhosa l1 test acc:", exp_l1['test_metrics']['accuracy'])
